In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(
    "../data/raw/SeoulBikeData.csv",
    encoding="ISO-8859-1"
)

# Create datetime index
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d/%m/%Y"
)

df["DateTime"] = (
    df["Date"] +
    pd.to_timedelta(df["Hour"], unit="h")
)

ts_df = df.sort_values("DateTime").set_index("DateTime")

# Extract hourly rental series
bike_series = ts_df["Rented Bike Count"].asfreq("h")

# Recreate chronological split
train = bike_series.iloc[:6132]
validation = bike_series.iloc[6132:7446]
test = bike_series.iloc[7446:]

print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Training: 6132
Validation: 1314
Test: 1314


In [3]:
import pandas as pd
import numpy as np

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Define feature creation
def create_features(series):

    X = pd.DataFrame(index=series.index)

    # Historical rental demand
    X["lag_1"] = series.shift(1)
    X["lag_2"] = series.shift(2)
    X["lag_24"] = series.shift(24)
    X["lag_48"] = series.shift(48)
    X["lag_168"] = series.shift(168)

    # Time-based features
    X["hour"] = series.index.hour
    X["day_of_week"] = series.index.dayofweek

    # Cyclical encoding of hour
    X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
    X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

    # Cyclical encoding of weekday
    X["day_sin"] = np.sin(2 * np.pi * X["day_of_week"] / 7)
    X["day_cos"] = np.cos(2 * np.pi * X["day_of_week"] / 7)

    # Weekend indicator
    X["is_weekend"] = (X["day_of_week"] >= 5).astype(int)

    return X


# Create features using the full chronological series.
# Every lag uses only past observations.
X_all = create_features(bike_series)

# Keep only training observations
X_train = X_all.loc[train.index]
y_train = train.copy()

# Remove rows without sufficient historical observations
valid_rows = X_train.notna().all(axis=1)

X_train = X_train.loc[valid_rows]
y_train = y_train.loc[valid_rows]

print("Training feature shape:", X_train.shape)
print("Training target shape:", y_train.shape)
print("\nFeatures:")
print(X_train.columns.tolist())

Training feature shape: (5964, 12)
Training target shape: (5964,)

Features:
['lag_1', 'lag_2', 'lag_24', 'lag_48', 'lag_168', 'hour', 'day_of_week', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend']


In [4]:
# Initialize scaler
scaler = StandardScaler()

# Fit scaler using training data only
X_train_scaled = scaler.fit_transform(X_train)

print("Original feature shape:", X_train.shape)
print("Scaled feature shape:", X_train_scaled.shape)

print("\nFirst 5 scaled observations:")
print(X_train_scaled[:5])

Original feature shape: (5964, 12)
Scaled feature shape: (5964, 12)

First 5 scaled observations:
[[-0.62241603 -0.41980251 -0.71511452 -0.79456368 -0.60971911 -1.65958328
   0.48985813 -0.00180115  1.41397648 -0.60118404 -1.27416239 -0.63869555]
 [-0.66292444 -0.62234158 -0.68394549 -0.79612212 -0.68810884 -1.51512003
   0.48985813  0.36422485  1.36578832 -0.60118404 -1.27416239 -0.63869555]
 [-0.66759849 -0.66284939 -0.79147862 -0.83820015 -0.73671047 -1.37065679
   0.48985813  0.70530678  1.22450778 -0.60118404 -1.27416239 -0.63869555]
 [-0.82184204 -0.66752337 -0.86472582 -0.89897953 -0.84018491 -1.22619355
   0.48985813  0.99820048  0.9997629  -0.60118404 -1.27416239 -0.63869555]
 [-0.87169854 -0.82176465 -0.92706386 -0.93014844 -0.88565096 -1.0817303
   0.48985813  1.22294571  0.70686968 -0.60118404 -1.27416239 -0.63869555]]


In [5]:
# Initialize SVR model
svr_model = SVR(
    kernel="rbf",
    C=100,
    epsilon=0.1,
    gamma="scale"
)

# Train the model
svr_model.fit(X_train_scaled, y_train)

print("SVR model trained successfully!")

print("\nModel parameters:")
print(svr_model.get_params())

SVR model trained successfully!

Model parameters:
{'C': 100, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.1, 'gamma': 'scale', 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}


In [6]:
# Start with all historical training observations
history = list(train.values)

# Store recursive predictions
svr_predictions = []

# Forecast each validation hour sequentially
for timestamp in validation.index:

    # Create features using historical values only
    features = pd.DataFrame([{
        "lag_1": history[-1],
        "lag_2": history[-2],
        "lag_24": history[-24],
        "lag_48": history[-48],
        "lag_168": history[-168],

        "hour": timestamp.hour,
        "day_of_week": timestamp.dayofweek,

        "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
        "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),

        "day_sin": np.sin(2 * np.pi * timestamp.dayofweek / 7),
        "day_cos": np.cos(2 * np.pi * timestamp.dayofweek / 7),

        "is_weekend": int(timestamp.dayofweek >= 5)
    }])

    # Ensure the same feature order as training
    features = features[X_train.columns]

    # Scale using the previously fitted scaler
    features_scaled = scaler.transform(features)

    # Predict the next hour
    prediction = svr_model.predict(features_scaled)[0]

    # Save prediction
    svr_predictions.append(prediction)

    # Add prediction to history for future forecasts
    history.append(prediction)


# Convert predictions into a time series
svr_predictions = pd.Series(
    svr_predictions,
    index=validation.index,
    name="SVR Forecast"
)

print("Number of forecasts:", len(svr_predictions))
print("\nFirst 10 predictions:")
print(svr_predictions.head(10))

Number of forecasts: 1314

First 10 predictions:
DateTime
2018-08-13 12:00:00     692.498134
2018-08-13 13:00:00     720.784072
2018-08-13 14:00:00     747.758284
2018-08-13 15:00:00     814.707841
2018-08-13 16:00:00     923.862493
2018-08-13 17:00:00    1095.768158
2018-08-13 18:00:00    1391.824053
2018-08-13 19:00:00    1529.646872
2018-08-13 20:00:00    1557.850083
2018-08-13 21:00:00    1495.498592
Freq: h, Name: SVR Forecast, dtype: float64


In [7]:
svr_mae = mean_absolute_error(
    validation,
    svr_predictions
)

svr_rmse = np.sqrt(
    mean_squared_error(validation, svr_predictions)
)

print("SVR VALIDATION RESULTS")
print("----------------------")
print("MAE:", svr_mae)
print("RMSE:", svr_rmse)

SVR VALIDATION RESULTS
----------------------
MAE: 498.2845103525345
RMSE: 753.7531456094338


In [8]:
# Scale the target variable using training data only
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
).ravel()

# Initialize SVR
svr_scaled_model = SVR(
    kernel="rbf",
    C=100,
    epsilon=0.1,
    gamma="scale"
)

# Train using scaled target
svr_scaled_model.fit(
    X_train_scaled,
    y_train_scaled
)

print("SVR with scaled target trained successfully!")
print("Scaled target mean:", y_train_scaled.mean())
print("Scaled target standard deviation:", y_train_scaled.std())

SVR with scaled target trained successfully!
Scaled target mean: -7.624871745245877e-17
Scaled target standard deviation: 1.0


In [9]:
history = list(train.values)
svr_scaled_predictions = []

for timestamp in validation.index:

    features = pd.DataFrame([{
        "lag_1": history[-1],
        "lag_2": history[-2],
        "lag_24": history[-24],
        "lag_48": history[-48],
        "lag_168": history[-168],

        "hour": timestamp.hour,
        "day_of_week": timestamp.dayofweek,

        "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
        "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),

        "day_sin": np.sin(2 * np.pi * timestamp.dayofweek / 7),
        "day_cos": np.cos(2 * np.pi * timestamp.dayofweek / 7),

        "is_weekend": int(timestamp.dayofweek >= 5)
    }])

    features = features[X_train.columns]

    features_scaled = scaler.transform(features)

    # Predict in standardized target scale
    prediction_scaled = svr_scaled_model.predict(
        features_scaled
    )[0]

    # Convert prediction back to rental counts
    prediction = y_scaler.inverse_transform(
        [[prediction_scaled]]
    )[0, 0]

    svr_scaled_predictions.append(prediction)

    # Use the prediction as historical input for the next hour
    history.append(prediction)


svr_scaled_predictions = pd.Series(
    svr_scaled_predictions,
    index=validation.index
)

# Evaluate
svr_scaled_mae = mean_absolute_error(
    validation,
    svr_scaled_predictions
)

svr_scaled_rmse = np.sqrt(
    mean_squared_error(
        validation,
        svr_scaled_predictions
    )
)

print("SVR WITH TARGET SCALING")
print("-----------------------")
print("MAE:", svr_scaled_mae)
print("RMSE:", svr_scaled_rmse)

SVR WITH TARGET SCALING
-----------------------
MAE: 488.7620770295194
RMSE: 759.9971280373111


In [10]:
svr_tuned_model = SVR(
    kernel="rbf",
    C=500,
    epsilon=0.2,
    gamma="scale"
)

svr_tuned_model.fit(
    X_train_scaled,
    y_train_scaled
)

print("Tuned SVR trained successfully!")

Tuned SVR trained successfully!


In [11]:
history = list(train.values)
svr_tuned_predictions = []

for timestamp in validation.index:

    features = pd.DataFrame([{
        "lag_1": history[-1],
        "lag_2": history[-2],
        "lag_24": history[-24],
        "lag_48": history[-48],
        "lag_168": history[-168],
        "hour": timestamp.hour,
        "day_of_week": timestamp.dayofweek,
        "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
        "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),
        "day_sin": np.sin(2 * np.pi * timestamp.dayofweek / 7),
        "day_cos": np.cos(2 * np.pi * timestamp.dayofweek / 7),
        "is_weekend": int(timestamp.dayofweek >= 5)
    }])

    features = features[X_train.columns]
    features_scaled = scaler.transform(features)

    prediction_scaled = svr_tuned_model.predict(
        features_scaled
    )[0]

    prediction = y_scaler.inverse_transform(
        [[prediction_scaled]]
    )[0, 0]

    svr_tuned_predictions.append(prediction)
    history.append(prediction)

svr_tuned_predictions = pd.Series(
    svr_tuned_predictions,
    index=validation.index
)

tuned_mae = mean_absolute_error(
    validation,
    svr_tuned_predictions
)

tuned_rmse = np.sqrt(
    mean_squared_error(
        validation,
        svr_tuned_predictions
    )
)

print("TUNED SVR RESULTS")
print("-----------------")
print("MAE:", tuned_mae)
print("RMSE:", tuned_rmse)

TUNED SVR RESULTS
-----------------
MAE: 851.745739438714
RMSE: 1119.7340641988108


In [12]:
# Save validation predictions
svr_results_df = pd.DataFrame({
    "Actual": validation,
    "SVR_Forecast": svr_scaled_predictions
})

svr_results_df.to_csv(
    "../results/tables/svr_validation_predictions.csv"
)

# Save SVR metrics
svr_metrics = pd.DataFrame({
    "Model": [
        "SVR Original Target",
        "SVR Scaled Target",
        "SVR Tuned C500"
    ],
    "MAE": [
        498.284510,
        488.762077,
        851.745739
    ],
    "RMSE": [
        753.753145,
        759.997128,
        1119.734064
    ]
})

svr_metrics.to_csv(
    "../results/tables/svr_validation_metrics.csv",
    index=False
)

print("SVR results saved successfully!")
print(svr_metrics)

SVR results saved successfully!
                 Model         MAE         RMSE
0  SVR Original Target  498.284510   753.753145
1    SVR Scaled Target  488.762077   759.997128
2       SVR Tuned C500  851.745739  1119.734064
